---

# DSC Gelar Rasa 2026: Klasifikasi Citra Sampah

*Label train dari clustering embedding DINOv3, nama cluster diputuskan tim dengan bukti visual, lalu satu classifier ConvNeXt-L DINOv3 untuk 8 kategori sampah.*

---

## Nama Tim

- Nama anggota 1
- Nama anggota 2
- Nama anggota 3

---

## Daftar Isi

1. [**Pendahuluan**](#1)
2. [**Inisialisasi**](#2)
3. [**Exploratory Data Analysis**](#3)
4. [**Ekstraksi Fitur**](#4)
5. [**Perbandingan Ruang Embedding**](#5)
6. [**Clustering Train**](#6)
7. [**Penamaan Cluster**](#7)
8. [**Training Classifier**](#8)
9. [**Validasi Lokal**](#9)
10. [**Inference Test**](#10)
11. [**Submission**](#11)


---

# Pendahuluan <a name="1"></a>

---

## Overview

Kompetisi meminta klasifikasi citra sampah ke 8 kategori: glass, paper, electronic, metal, plastic, cardboard, textile, dan foodwaste. Data train berisi 1.920 citra tanpa label dan data test berisi 480 citra. Menurut jawaban resmi panitia, label train wajib diperoleh lewat clustering, lalu cluster boleh dinamai secara manual.

## Metrik

Submission dinilai dengan accuracy, yaitu porsi prediksi benar terhadap seluruh citra test. Public leaderboard memuat sekitar 30% test (sekitar 145 citra), sehingga satu kesalahan bernilai sekitar 0,7 poin persen. Notebook ini tidak memakai skor leaderboard untuk keputusan apa pun.

## Kepatuhan

| Aturan | Penerapan di notebook |
|---|---|
| Tanpa VLM | Hanya DINOv3 (self-supervised, citra saja) dan ConvNeXtV2 ImageNet-1k (citra saja) |
| Label train dari clustering | KMeans pada embedding train, nama cluster diisi tim di bagian 7 |
| Test hanya untuk inference | Test tidak ikut PCA, clustering, grup, normalisasi, atau seleksi model |
| Satu model | Submission dari satu ConvNeXt-L yang dilatih ulang pada seluruh label, tanpa ensemble dan tanpa TTA |
| Tanpa LB probing | Semua keputusan dari aturan yang ditetapkan sebelum run dan dari validasi lokal |

## Pendekatan

1. EDA ringkas: resolusi, kualitas JPEG, noise dan blur, warna, serta duplikat.
2. Embedding train dari DINOv3 ViT-7B (bobot fp16 dibagi ke dua T4) dan DINOv3 ViT-L sebagai pembanding.
3. Ruang embedding dipilih dengan aturan beku berbasis metrik tanpa label.
4. KMeans train-only, stabilitas per citra dari bootstrap, citra rapuh tidak diberi label.
5. Tim menamai tiap cluster dari montage dan top-5 kelas ImageNet sebagai bukti pendukung.
6. ConvNeXt-L DINOv3 di-fine-tune, divalidasi dengan StratifiedGroupKFold berbasis grup near-duplicate.
7. Model final dilatih ulang pada seluruh label, lalu memprediksi test sekali.

Embedding 7B hanya dibutuhkan untuk train. Classifier final membaca piksel, jadi test tidak perlu diekstrak dengan 7B.

---

# Inisialisasi <a name="2"></a>

---

## Environment Setup

Notebook dirancang untuk Kaggle GPU T4 x2 dengan Internet aktif. Bila dataset cache embedding dan cluster dilampirkan, `Settings.PAKAI_CACHE = True` melewati ekstraksi 7B. Perkiraan waktu: ekstraksi 7B train dua pandangan sekitar 30 menit, cross-validation 5 fold sekitar 1 jam, model final sekitar 15 menit.

In [ ]:
!nvidia-smi

Sel berikut memasang pustaka yang versinya perlu dikunci.

In [ ]:
%pip install -q timm==1.0.29

## Import Libraries

Semua import dikumpulkan di satu sel.

In [ ]:
import gc
import hashlib
import json
import math
import os
import random
import shutil
import time
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import timm
import torch
import torch.nn as nn
from accelerate import init_empty_weights
from accelerate.utils import set_module_tensor_to_device
from huggingface_hub import hf_hub_download
from IPython.display import display
from kaggle_secrets import UserSecretsClient
from PIL import Image, ImageOps
from safetensors import safe_open
from scipy.fft import dctn
from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import connected_components
from scipy.stats import ks_2samp
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import adjusted_rand_score, confusion_matrix, silhouette_score
from sklearn.model_selection import StratifiedGroupKFold
from timm.data import ImageNetInfo
from timm.optim import create_optimizer_v2
from torch.utils.data import DataLoader, Dataset
from torchvision.transforms import v2 as T

print('timm', timm.__version__, '| torch', torch.__version__, '| GPU', torch.cuda.device_count())

## Seed Everything

In [ ]:
def seed_everything(seed: int = 42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True


def seed_worker(worker_id):
    s = torch.initial_seed() % 2**32
    np.random.seed(s)
    random.seed(s)


seed_everything(42)

## Settings

Semua path, revisi bobot, ambang, dan hyperparameter dikumpulkan di sini. Token Hugging Face dibaca dari Kaggle Secrets bernama `HF_TOKEN` bila dilampirkan. Token tidak wajib karena repo bobot tidak gated, tetapi mempercepat unduhan dan menghindari batas laju. Parameter augmentasi berasal dari verifikasi preprocessing lokal (`temp/prep_301_verifikasi`), bukan dari skor test. Aturan pemilihan ruang embedding, K, dan ambang stabilitas ditetapkan sebelum run.

In [ ]:
class Settings:
    SEED = 42
    KELAS = ['glass', 'paper', 'electronic', 'metal', 'plastic', 'cardboard', 'textile', 'foodwaste']
    INPUT = Path('/kaggle/input')
    OUTPUT_DIR = Path('/kaggle/working')
    CACHE_DIR = OUTPUT_DIR / 'cache'
    FIG_DIR = OUTPUT_DIR / 'gambar'
    HF_CACHE = Path('/tmp/hf')
    PAKAI_CACHE = True
    ULANG_CV = False

    M7B = ('timm/vit_7b_patch16_dinov3.lvd1689m', 'b40f6c827cb02b13c7a23a76cf598489bb0f94e5', 'vit_7b_patch16_dinov3')
    MVITL = ('timm/vit_large_patch16_dinov3.lvd1689m', '30c1109559f65dea34316b0d4842d35c5771fe11', 'vit_large_patch16_dinov3')
    MCNN = ('timm/convnext_large.dinov3_lvd1689m', '0f8b496db5dea10c7533e85304400f0eddad55ac', 'convnext_large.dinov3_lvd1689m')
    MHEAD = ('timm/convnextv2_huge.fcmae_ft_in22k_in1k_384', 'e2ddce36b729b827f01e8c3fd72be459d212c95c', 'convnextv2_huge.fcmae_ft_in22k_in1k_384')
    POLA_7B_FP16 = 'dinov3_7b_fp16*.safetensors'
    DISK_MIN_GIB = 30

    UKURAN_SIMPAN = 512
    UKURAN = 384
    MEAN = (0.485, 0.456, 0.406)
    STD = (0.229, 0.224, 0.225)
    BS_EMB = 16
    LAPIS_7B = (19, 29)

    PCA_DIM = 128
    K_BANDING = 48
    K_KANDIDAT = (32, 48, 64)
    UK_MIN_CLUSTER = 10
    N_BOOT = 20
    STAB_MIN = 0.7
    SELISIH_ARI_7B = 0.03
    H_MAX = 4
    PORSI_PASANGAN = 0.000721

    N_FOLD = 5
    EPOCH = 8
    BS = 16
    LR = 1e-4
    HEAD_MULT = 10.0
    LAYER_DECAY = 0.75
    WD = 0.05
    WARMUP = 1
    SMOOTH = 0.1
    DROP_PATH = 0.1
    RRC_SCALE = (0.4, 1.0)
    JITTER = (0.2, 0.1)
    TRIM = (12, 4, 0.03)
    NUM_WORKERS = 4


CFG = Settings()
for d in (CFG.CACHE_DIR, CFG.FIG_DIR, CFG.HF_CACHE):
    d.mkdir(parents=True, exist_ok=True)
try:
    os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
    print('HF_TOKEN dari Kaggle Secrets dipakai')
except Exception:
    print('HF_TOKEN tidak terpasang; repo timm DINOv3 tidak gated, unduhan tetap jalan tanpa token')

## Load Dataset

Lokasi data dicari dari berkas `SampleSubmission_DSC.csv`, sehingga nama folder dataset tidak perlu ditulis tangan. Nama berkas diurutkan sebagai angka karena ekstensinya campur (`.jpg`, `.jpeg`, `.png`).

In [ ]:
SUB_PATH = next(CFG.INPUT.rglob('SampleSubmission_DSC.csv'))
TRAIN_DIR = next(p for p in SUB_PATH.parent.rglob('train') if p.is_dir())
TEST_DIR = next(p for p in SUB_PATH.parent.rglob('test') if p.is_dir())
f_train = sorted(TRAIN_DIR.iterdir(), key=lambda p: int(p.stem))
f_test = sorted(TEST_DIR.iterdir(), key=lambda p: int(p.stem))
id_train = np.array([int(p.stem) for p in f_train])
id_test = np.array([int(p.stem) for p in f_test])
assert len(f_train) == 1920 and len(f_test) == 480, (len(f_train), len(f_test))
print(TRAIN_DIR, len(f_train), '|', TEST_DIR, len(f_test))

---

# Exploratory Data Analysis <a name="3"></a>

---

EDA di notebook ini adalah versi ringkas dari EDA lokal (`temp/eda_201` sampai `temp/eda_209`), yang memakai seluruh fitur berkas, warna, latar, duplikat, outlier, dan shift. Piksel test tidak dibaca di bagian ini. Dari test hanya header berkas (ukuran dan tabel kuantisasi JPEG) yang dipakai, dan itu pun hanya sebagai diagnostik.

## Resolusi dan Format

Setiap citra train dibuka sekali: orientasi EXIF diterapkan, mode dikonversi ke RGB (ada PNG mode P dan RGBA), pita hitam murni dari sumber dipotong, lalu disimpan di RAM sebagai 512 x 512. Ukuran 512 cukup untuk crop acak sebelum resize ke 384. Kualitas JPEG ditaksir dari tabel kuantisasi luminans terhadap tabel standar IJG.

In [ ]:
IJG_LUMA = np.array([16, 11, 10, 16, 24, 40, 51, 61, 12, 12, 14, 19, 26, 58, 60, 55, 14, 13, 16, 24, 40, 57, 69, 56,
                     14, 17, 22, 29, 51, 87, 80, 62, 18, 22, 37, 56, 68, 109, 103, 77, 24, 35, 55, 64, 81, 104, 113, 92,
                     49, 64, 78, 87, 103, 121, 120, 101, 72, 92, 95, 98, 112, 100, 103, 99], float)


def kualitas_jpeg(im):
    q = getattr(im, 'quantization', None)
    if not q:
        return np.nan
    s = np.mean(q[0]) / IJG_LUMA.mean() * 100
    return (200 - s) / 2 if s <= 100 else 5000 / s


def header(p):
    with Image.open(p) as im:
        return dict(w=im.width, h=im.height, fmt=im.format, mode=im.mode, q=kualitas_jpeg(im))


def trim_hitam(a, ambang=CFG.TRIM[0], std_maks=CFG.TRIM[1], porsi_min=CFG.TRIM[2]):
    g = a.max(2).astype(np.float32)

    def tepi(pmax, pstd):
        n = 0
        while n < len(pmax) and pmax[n] < ambang and pstd[n] < std_maks:
            n += 1
        return n

    rmax, rstd, cmax, cstd = np.percentile(g, 99, axis=1), g.std(1), np.percentile(g, 99, axis=0), g.std(0)
    t, b, l, r = tepi(rmax, rstd), tepi(rmax[::-1], rstd[::-1]), tepi(cmax, cstd), tepi(cmax[::-1], cstd[::-1])
    H, W = g.shape
    t, b = (t, b) if (t + b) / H >= porsi_min else (0, 0)
    l, r = (l, r) if (l + r) / W >= porsi_min else (0, 0)
    if H - t - b < H * .3 or W - l - r < W * .3:
        return a
    return a[t:H - b, l:W - r]


def muat_citra(p):
    with Image.open(p) as im:
        a = trim_hitam(np.asarray(ImageOps.exif_transpose(im).convert('RGB')))
    return np.asarray(Image.fromarray(a).resize((CFG.UKURAN_SIMPAN, CFG.UKURAN_SIMPAN), Image.BICUBIC))


with ThreadPoolExecutor(8) as ex:
    X_train = np.stack(list(ex.map(muat_citra, f_train)))
meta = pd.concat([pd.DataFrame([header(p) for p in f_train]).assign(split='train'),
                  pd.DataFrame([header(p) for p in f_test]).assign(split='test')], ignore_index=True)
meta['pendek'] = meta[['w', 'h']].min(axis=1)
meta['ar'] = meta.w / meta.h
ringkas = meta.groupby('split').agg(n=('w', 'size'), pendek_median=('pendek', 'median'),
                                    pendek_lt384=('pendek', lambda s: (s < 384).mean()),
                                    persegi=('ar', lambda s: (abs(s - 1) < .02).mean()), q_median=('q', 'median'))
display(ringkas.round(3))
display(meta.groupby(['split', 'fmt', 'mode']).size().rename('n').reset_index())
for c in ('pendek', 'ar', 'q'):
    r = ks_2samp(meta.loc[meta.split == 'train', c].dropna(), meta.loc[meta.split == 'test', c].dropna())
    print(f'KS {c}: D={r.statistic:.3f} (diagnostik shift, bukan untuk tuning)')

fig, ax = plt.subplots(1, 2, figsize=(12, 3.5))
for s, c in (('train', 'C0'), ('test', 'C3')):
    d = meta[meta.split == s]
    ax[0].hist(np.log2(d.pendek), 40, alpha=.5, density=True, color=c, label=s)
    ax[1].hist(np.clip(d.ar, .3, 2.5), 50, alpha=.5, density=True, color=c, label=s)
ax[0].axvline(np.log2(CFG.UKURAN), c='k', ls='--')
ax[0].set(title='log2 sisi pendek (garis: 384)')
ax[1].set(title='aspect ratio w/h')
for a in ax:
    a.legend()
plt.tight_layout()
plt.show()

#### Insights

> Median sisi pendek 416 px, 88% train berukuran minimal 384 px, dan 77% train persegi (test 82%). Input 384 membuat hampir semua citra di-downscale. Input 448 akan meng-upsample sekitar 53% citra (ukuran 416), dan ini sejalan dengan catatan lama bahwa ConvNeXt 448 lebih buruk dari 384.

> Center crop memotong lebih dari 10% box objek pada 16,7% citra (EDA lokal `prep_301`). Letterbox menjaga bentuk, tetapi grid visual memperlihatkan masalah lain: 123 citra train (113 berlabel lama electronic) dan 45 citra test membawa pita hitam letterbox dari sumbernya. Pita ini adalah jalan pintas menuju kelas electronic dan memboroskan rata-rata 30% luas citra. Pita hitam murni dipotong lebih dulu. Setelah dipotong, citra dengan distorsi di atas 1,34 masih 31,5% electronic (basis 10,1%), sehingga letterbox akan memunculkan pita lagi pada kelas yang sama. Karena itu resize memakai stretch bicubic ke persegi, dan distorsi di atas 1,5 hanya terjadi pada 7,9% citra.

> Sebanyak 87% train dan 94% test sudah di-encode ulang di kualitas sekitar 75 oleh encoder yang sama. Perbedaan sumber berkas antar split adalah shift utama: AUC adversarial pada metadata 0,66 dan pada embedding 0,65, keduanya hanya diagnostik.

## Noise, Blur, Warna, dan Latar

Noise ditaksir dengan metode Immerkaer, blur dengan variance Laplacian pada skala abu-abu 256 px, warna dari rata-rata Lab area pusat 50%, dan keseragaman latar dari simpangan baku Lab cincin tepi 6%.

In [ ]:
K_IMMERKAER = np.array([[1, -2, 1], [-2, 4, -2], [1, -2, 1]], np.float32)


def statistik(x):
    g = cv2.cvtColor(x, cv2.COLOR_RGB2GRAY).astype(np.float32) / 255
    noise = np.abs(cv2.filter2D(g, -1, K_IMMERKAER)).mean() * math.sqrt(math.pi / 2) / 6
    blur = cv2.Laplacian(cv2.resize(g, (256, 256), interpolation=cv2.INTER_AREA), cv2.CV_32F).var()
    lab = cv2.cvtColor(cv2.resize(x, (128, 128), interpolation=cv2.INTER_AREA), cv2.COLOR_RGB2LAB).astype(np.float32)
    lab[..., 0] *= 100 / 255
    lab[..., 1:] -= 128
    pusat = lab[32:96, 32:96].reshape(-1, 3)
    tepi = np.concatenate([lab[:8].reshape(-1, 3), lab[-8:].reshape(-1, 3), lab[:, :8].reshape(-1, 3), lab[:, -8:].reshape(-1, 3)])
    return dict(noise=noise, blur=blur, L=pusat[:, 0].mean(), chroma=np.hypot(pusat[:, 1], pusat[:, 2]).mean(),
                std_tepi=np.linalg.norm(tepi.std(0)), L_tepi=np.median(tepi[:, 0]))


stat = pd.DataFrame([statistik(x) for x in X_train])
stat['latar'] = np.where(stat.std_tepi >= 8, 'bertekstur', np.where(stat.L_tepi > 85, 'putih_polos', 'polos_lain'))
print(f"noise > 0,03: {(stat.noise > .03).mean():.3f} | blur < 5e-4: {(stat.blur < 5e-4).mean():.3f}")
print(stat.latar.value_counts(normalize=True).round(3).to_dict())
fig, ax = plt.subplots(1, 3, figsize=(15, 3.5))
ax[0].hist(np.log10(stat.noise + 1e-5), 40)
ax[0].axvline(np.log10(.03), c='k', ls='--')
ax[0].set(title='log10 noise')
ax[1].hist(np.log10(stat.blur + 1e-6), 40)
ax[1].axvline(np.log10(5e-4), c='k', ls='--')
ax[1].set(title='log10 variance Laplacian')
ax[2].scatter(stat.L, stat.chroma, s=3, alpha=.4)
ax[2].set(xlabel='L* pusat', ylabel='chroma pusat', title='warna area pusat')
plt.tight_layout()
plt.show()

#### Insights

> Pada resolusi asli (EDA lokal), taksiran noise tinggi hanya sekitar 2,4% dan blur sekitar 2%, terkumpul di citra hasil unduhan web. Angka di atas lebih kecil karena dihitung pada cache 512 px, dan downscale meredam noise. Inspeksi visual menunjukkan citra dengan noise tertinggi sebenarnya adalah tumpukan pakaian atau sampah yang teksturnya padat, beberapa dengan artefak blok upscale dan watermark situs stok foto. Jadi estimator Immerkaer di sini lebih banyak mengukur tekstur daripada noise sensor. Denoise NLM h=6 menghaluskan tumpukan kain dan menghapus 17% energi tekstur pada citra bersih, padahal tekstur adalah isyarat material. Tidak ada denoise.

> Cardboard dibedakan dari paper terutama oleh chroma (b* median 20,8 lawan 7,2), sedangkan glass, metal, paper, dan plastic hanya berjarak 3,6 sampai 7 dE. Augmentasi warna yang menurunkan chroma berisiko membuat cardboard mirip paper.

> Sekitar 77% citra berlatar bertekstur dan 17% berlatar putih polos. Porsi latar putih berbeda per cluster (sampai 52% pada cluster yang didominasi kaca), jadi cluster bisa ikut menangkap latar. Montage di bagian 7 perlu diperiksa untuk pola ini.

## Near-duplicate pHash

pHash 63 bit dihitung pada 8 transformasi dihedral karena data memuat salinan hasil flip dan rotasi 90 derajat. Jarak Hamming minimum atas 8 transformasi dipakai. Ambang 4 dipilih di EDA lokal: kesepakatan label lama pada pasangan Hamming 4 atau kurang 0,99, sedangkan pada ambang 10 hanya 0,64 karena citra berlatar polos merantai.

In [ ]:
def phash8(x):
    g = cv2.resize(cv2.cvtColor(x, cv2.COLOR_RGB2GRAY), (32, 32), interpolation=cv2.INTER_AREA).astype(np.float32)
    out = []
    for b in (g, np.fliplr(g)):
        for r in range(4):
            d = dctn(np.rot90(b, r), norm='ortho')[:8, :8].ravel()[1:]
            out.append(d > np.median(d))
    return np.array(out)


PH = np.stack([phash8(x) for x in X_train])
basis = PH[:, 0, :].astype(np.float32) * 2 - 1
HAM = np.min([(63 - (PH[:, t, :].astype(np.float32) * 2 - 1) @ basis.T) / 2 for t in range(8)], axis=0)
HAM = np.minimum(HAM, HAM.T)
iu = np.triu_indices(len(HAM), 1)
for h in (0, 2, 4, 10):
    print(f'pasangan train-train Hamming <= {h}: {int((HAM[iu] <= h).sum())}')

#### Insights

> EDA lokal menemukan 100 pasangan train dengan Hamming 4 atau kurang dan 7,3% citra test punya pasangan train pada ambang yang sama. Split validasi acak akan menaruh salinan di train dan validasi sekaligus, jadi validasi memakai StratifiedGroupKFold dengan grup near-duplicate (bagian 6).

## Verifikasi Augmentasi

Parameter augmentasi dipilih lewat iterasi lokal (`temp/prep_301_verifikasi`) dengan proksi tanpa model. RandomResizedCrop scale minimum 0,4 adalah nilai terkecil dengan peluang sisa objek di bawah 50% tidak lebih dari 1% (diukur pada citra berlatar polos). Rotasi 90 derajat dan flip meniru salinan dihedral yang memang ada di data. Rotasi kecil 15 derajat ditolak setelah inspeksi grid: isian sudut konstan mencolok, dan isian warna tepi memunculkan sudut hitam pada citra electronic berpita. Color jitter dibatasi pada brightness 0,2 dan contrast 0,1 tanpa saturation dan hue. Jitter lama (0,2; 0,2; 0,2; 0,02) memindahkan 4,7% cardboard ke sisi paper, sedangkan parameter final 1,7%. Grid di bawah memperlihatkan hasil augmentasi untuk pemeriksaan visual.

In [ ]:
AUG = [
    T.RandomResizedCrop(CFG.UKURAN, scale=CFG.RRC_SCALE, ratio=(3 / 4, 4 / 3), antialias=True),
    T.RandomHorizontalFlip(),
    T.RandomVerticalFlip(),
    T.RandomChoice([T.RandomRotation((a, a)) for a in (0, 90, 180, 270)]),
    T.ColorJitter(brightness=CFG.JITTER[0], contrast=CFG.JITTER[1]),
]
NORMAL = [T.ToDtype(torch.float32, scale=True), T.Normalize(CFG.MEAN, CFG.STD)]
tf_train = T.Compose([T.ToImage(), *AUG, *NORMAL])
tf_eval = T.Compose([T.ToImage(), T.Resize((CFG.UKURAN, CFG.UKURAN), antialias=True), *NORMAL])
tf_tampil = T.Compose([T.ToImage(), *AUG])

rng = np.random.default_rng(CFG.SEED)
fig, ax = plt.subplots(6, 4, figsize=(8, 12))
for r, i in enumerate(rng.choice(len(X_train), 6, replace=False)):
    ax[r, 0].imshow(X_train[i])
    ax[r, 0].set_title(f'{id_train[i]} asli', fontsize=7)
    for c in range(1, 4):
        ax[r, c].imshow(tf_tampil(X_train[i]).permute(1, 2, 0).numpy())
        ax[r, c].set_title('augmentasi', fontsize=7)
for a in ax.ravel():
    a.axis('off')
plt.tight_layout()
plt.show()

#### Insights

> Augmentasi menjaga isyarat material: crop tidak lebih kecil dari 40% luas, warna kardus tidak dipudarkan lewat saturation, tekstur tidak dihaluskan, dan tidak ada sudut isian yang bisa menjadi artefak. Grid lengkap per cluster lama (resize, trim pita hitam, crop, rotasi, jitter final, serta contoh parameter yang ditolak) ada di `temp/prep_301_verifikasi/output` untuk diperiksa tim.

---

# Ekstraksi Fitur <a name="4"></a>

---

Embedding train diambil dari DINOv3 ViT-7B dan DINOv3 ViT-L dengan input dan preprocessing yang sama. Fitur yang disimpan untuk tiap pandangan (asli dan flip horizontal): token CLS lapis akhir, rata-rata patch token lapis akhir, dan, khusus 7B, CLS lapis 20 dan 30. Semua array disimpan fp16 di `cache/` beserta manifest (revisi bobot, waktu, VRAM, hash), sehingga run berikutnya cukup memuat cache.

## Utilitas Bobot dan Cache

Bobot diunduh dengan revisi yang dikunci. Bobot 7B dicari lebih dulu sebagai dataset fp16 terlampir (`dinov3_7b_fp16*.safetensors`, sekitar 13,4 GB). Bila tidak ada, berkas fp32 resmi timm (26,9 GB, tidak gated) diunduh ke `/tmp`, dengan pemeriksaan ruang disk lebih dulu.

In [ ]:
def unduh(repo, rev, nama='model.safetensors'):
    return Path(hf_hub_download(repo, nama, revision=rev, cache_dir=CFG.HF_CACHE))


def cari_cache(nama):
    for akar in (CFG.CACHE_DIR, CFG.INPUT):
        hit = sorted(akar.rglob(nama))
        if hit:
            return hit[0]
    return None


def berkas_7b():
    lokal = sorted(CFG.INPUT.rglob(CFG.POLA_7B_FP16))
    if lokal:
        return lokal
    bebas = shutil.disk_usage(CFG.HF_CACHE).free / 2**30
    assert bebas >= CFG.DISK_MIN_GIB, f'Disk {bebas:.0f} GiB tidak cukup untuk bobot fp32 26,9 GB. Lampirkan dataset bobot fp16.'
    return [unduh(*CFG.M7B[:2])]


class Citra(Dataset):
    def __init__(self, X, tf, idx=None, y=None):
        self.X, self.tf, self.y = X, tf, y
        self.idx = np.arange(len(X)) if idx is None else np.asarray(idx)

    def __len__(self):
        return len(self.idx)

    def __getitem__(self, i):
        x = self.tf(self.X[self.idx[i]])
        return x if self.y is None else (x, int(self.y[i]))

## DINOv3 di Dua GPU

Bobot 7B fp16 sekitar 13,4 GB. Satu T4 (sekitar 15 GB terpakai) hanya menyisakan sekitar 1 GB untuk aktivasi, jadi 20 blok pertama ditaruh di GPU 0 dan 20 blok berikutnya di GPU 1. Model dibuat kosong dengan `init_empty_weights`, lalu tiap tensor dibaca satu per satu dari safetensors, dikonversi ke fp16, dan langsung dikirim ke GPU tujuan. Dengan cara ini RAM tidak pernah memuat seluruh bobot fp32.

T4 tidak mendukung bf16, padahal DINOv3 dilatih dengan bf16. Untuk mencegah overflow fp16, residual stream disimpan dalam fp32, sedangkan matmul di attention dan MLP berjalan di fp16 lewat autocast. Bila sebuah blok tetap menghasilkan nilai non-finite, blok itu otomatis dijalankan ulang dalam fp32. `cek_presisi` membandingkan keluaran tiap blok fp16 dengan fp32 pada 4 citra sebelum ekstraksi.

In [ ]:
def muat_dinov3(arsitektur, berkas, dua_gpu):
    with init_empty_weights():
        m = timm.create_model(arsitektur, pretrained=False, num_classes=0)
    n = len(m.blocks)
    batas = n // 2 if dua_gpu and torch.cuda.device_count() > 1 else n
    akhir = 'cuda:1' if batas < n else 'cuda:0'

    def perangkat(k):
        if k.startswith('blocks.'):
            return 'cuda:0' if int(k.split('.')[1]) < batas else 'cuda:1'
        return akhir if k.startswith(('norm.', 'fc_norm.', 'head.')) else 'cuda:0'

    param = dict(m.named_parameters())
    kunci_model = set(m.state_dict())
    terisi, asing = set(), []
    for f in berkas:
        with safe_open(str(f), 'pt', device='cpu') as sf:
            for k in sf.keys():
                if k not in kunci_model:
                    asing.append(k)
                    continue
                t = sf.get_tensor(k)
                fp16 = k in param and t.is_floating_point()
                set_module_tensor_to_device(m, k, perangkat(k), value=t, dtype=torch.float16 if fp16 else None)
                terisi.add(k)
    kurang = set(param) - terisi
    assert not kurang, f'parameter tanpa bobot: {sorted(kurang)[:5]}'
    for nama, b in m.named_buffers():
        if b.device.type != 'cuda':
            set_module_tensor_to_device(m, nama, perangkat(nama), value=b)
    m.eval()
    m.batas = batas
    m.blok32 = set()
    print(f'{arsitektur}: {len(terisi)} tensor, {len(asing)} kunci asing dilewati, blok GPU0 {batas}, GPU1 {n - batas}')
    return m


def awal(m, x):
    with torch.autocast('cuda', dtype=torch.float16):
        h = m.patch_embed(x.to('cuda:0'))
        h, rope = m._pos_embed(h)
        h = m.norm_pre(h)
    return h.float(), rope


def lewat_blok(m, i, h, rope):
    d = 'cuda:0' if i < m.batas else 'cuda:1'
    h = h.to(d)
    r = None if rope is None else rope.to(d)
    blk = m.blocks[i]
    if i not in m.blok32:
        with torch.autocast('cuda', dtype=torch.float16):
            y = blk(h, rope=r)
        if torch.isfinite(y).all():
            return y.float()
        blk.float()
        m.blok32.add(i)
    with torch.autocast('cuda', enabled=False):
        return blk(h, rope=None if r is None else r.float()).float()


def normal_akhir(m, h):
    with torch.autocast('cuda', dtype=torch.float16):
        return m.norm(h.to(m.norm.weight.device)).float()


@torch.no_grad()
def maju_dinov3(m, x, lapis=()):
    h, rope = awal(m, x)
    antara = {}
    for i in range(len(m.blocks)):
        h = lewat_blok(m, i, h, rope)
        if i in lapis:
            antara[i] = normal_akhir(m, h)[:, 0]
    h = normal_akhir(m, h)
    return h[:, 0], h[:, m.num_prefix_tokens:].mean(1), antara


@torch.no_grad()
def cek_presisi(m, x):
    h, rope = awal(m, x)
    rows = []
    for i, blk in enumerate(m.blocks):
        d = 'cuda:0' if i < m.batas else 'cuda:1'
        h = h.to(d)
        r = None if rope is None else rope.to(d)
        with torch.autocast('cuda', dtype=torch.float16):
            y16 = blk(h, rope=r).float()
        blk.float()
        with torch.autocast('cuda', enabled=False):
            y32 = blk(h, rope=None if r is None else r.float()).float()
        if i not in m.blok32:
            blk.half()
        rows.append(dict(blok=i, galat_relatif=((y16 - y32).norm() / y32.norm()).item(),
                         maks_abs=y32.abs().max().item(), finite16=bool(torch.isfinite(y16).all())))
        h = y32
    return pd.DataFrame(rows)

## Ekstraksi Embedding Train

Ekstraksi memakai `tf_eval` (stretch ke 384, tanpa augmentasi). Hanya train yang diekstrak. Pada 384 px, satu citra menghasilkan 576 patch, 1 CLS, dan 4 register. Perkiraan komputasi 7B sekitar 8 TFLOP per citra per pandangan, jadi 1.920 citra x 2 pandangan butuh sekitar 30 menit di T4 karena blok GPU 0 dan GPU 1 bergantian.

In [ ]:
@torch.no_grad()
def ekstrak(m, X, lapis=()):
    dl = DataLoader(Citra(X, tf_eval), batch_size=CFG.BS_EMB, num_workers=CFG.NUM_WORKERS)
    out = {}
    for x in dl:
        for v, xx in (('asli', x), ('flip', x.flip(3))):
            cls, patch, antara = maju_dinov3(m, xx, lapis)
            out.setdefault(f'{v}_cls', []).append(cls.half().cpu())
            out.setdefault(f'{v}_patch', []).append(patch.half().cpu())
            for i, a in antara.items():
                out.setdefault(f'{v}_l{i + 1}', []).append(a.half().cpu())
    return {k: torch.cat(v).numpy() for k, v in out.items()}


def embedding(nama, arsitektur, berkas_fn, dua_gpu, lapis=()):
    kunci = [f'{v}_{f}' for v in ('asli', 'flip') for f in ['cls', 'patch'] + [f'l{i + 1}' for i in lapis]]
    nama_berkas = {k: f'emb_{nama}_train_{k}.npy' for k in kunci}
    if CFG.PAKAI_CACHE and all(cari_cache(b) for b in nama_berkas.values()):
        print(nama, 'dimuat dari cache', cari_cache(nama_berkas[kunci[0]]).parent)
        return {k: np.load(cari_cache(b)).astype(np.float32) for k, b in nama_berkas.items()}
    for i in range(torch.cuda.device_count()):
        torch.empty(0, device=f'cuda:{i}')
        torch.cuda.reset_peak_memory_stats(i)
    t0 = time.time()
    m = muat_dinov3(arsitektur, berkas_fn(), dua_gpu)
    t_muat = time.time() - t0
    presisi = cek_presisi(m, torch.stack([tf_eval(X_train[i]) for i in range(4)]))
    presisi.to_csv(CFG.CACHE_DIR / f'presisi_{nama}.csv', index=False)
    print(f'{nama}: galat relatif fp16 vs fp32 maks {presisi.galat_relatif.max():.2e}, '
          f'blok non-finite di fp16 {int((~presisi.finite16).sum())}, |aktivasi| maks {presisi.maks_abs.max():.0f}')
    t1 = time.time()
    hasil = ekstrak(m, X_train, lapis)
    manifest = dict(model=nama, arsitektur=arsitektur, ukuran=CFG.UKURAN, presisi='bobot fp16, residual fp32',
                    blok_fp32=sorted(m.blok32), detik_muat=round(t_muat), detik_ekstrak=round(time.time() - t1),
                    vram_gib=[round(torch.cuda.max_memory_allocated(i) / 2**30, 2) for i in range(torch.cuda.device_count())],
                    timm=timm.__version__, sha256={k: hashlib.sha256(a.tobytes()).hexdigest()[:16] for k, a in hasil.items()})
    for k, a in hasil.items():
        np.save(CFG.CACHE_DIR / nama_berkas[k], a)
    json.dump(manifest, open(CFG.CACHE_DIR / f'manifest_{nama}.json', 'w'), indent=1)
    print(manifest)
    del m
    gc.collect()
    torch.cuda.empty_cache()
    return {k: a.astype(np.float32) for k, a in hasil.items()}


E_VITL = embedding('vitl', CFG.MVITL[2], lambda: [unduh(*CFG.MVITL[:2])], dua_gpu=False)
E_7B = embedding('7b', CFG.M7B[2], berkas_7b, dua_gpu=True, lapis=CFG.LAPIS_7B)
print({k: v.shape for k, v in E_7B.items()})

## Head ImageNet-1k sebagai Bukti Pendukung

ConvNeXtV2-Huge (pretraining FCMAE, fine-tune ImageNet-1k, citra saja) memberi probabilitas 1.000 kelas ImageNet untuk tiap citra train. Keluarannya dipakai untuk dua hal: referensi independen pada perbandingan ruang embedding, dan daftar top-5 nama kelas ImageNet per cluster sebagai bukti pendukung penamaan. Tidak ada pemetaan kelas ImageNet ke 8 kategori. Nama mentah ditampilkan, dan tim yang menafsirkannya.

In [ ]:
def prob_head():
    b = 'head_in1k_train.npy'
    if CFG.PAKAI_CACHE and cari_cache(b):
        return np.load(cari_cache(b)).astype(np.float32)
    m = timm.create_model(CFG.MHEAD[2], pretrained=True, pretrained_cfg_overlay={'file': str(unduh(*CFG.MHEAD[:2]))})
    m = m.cuda().eval().half()
    dl = DataLoader(Citra(X_train, tf_eval), batch_size=32, num_workers=CFG.NUM_WORKERS)
    with torch.no_grad(), torch.autocast('cuda', dtype=torch.float16):
        p = np.concatenate([torch.softmax(m(x.cuda()).float(), 1).cpu().numpy() for x in dl])
    np.save(CFG.CACHE_DIR / b, p.astype(np.float16))
    del m
    gc.collect()
    torch.cuda.empty_cache()
    return p


P_HEAD = prob_head()
INFO_IN = ImageNetInfo()
NAMA_IN = [INFO_IN.index_to_description(i) for i in range(1000)]
TOP1 = P_HEAD.argmax(1)
print('kelas ImageNet top-1 terbanyak:', pd.Series([NAMA_IN[i].split(',')[0] for i in TOP1]).value_counts().head(8).to_dict())

---

# Perbandingan Ruang Embedding <a name="5"></a>

---

Bagian ini menjawab apakah 7B benar-benar memberi cluster yang lebih baik daripada ViT-L. Resep fitur sama untuk keduanya: CLS dan rata-rata patch, masing-masing dirata-rata atas pandangan asli dan flip, dinormalisasi L2, digabung, lalu PCA 128 yang di-fit pada train saja.

| Metrik | Arti | Bisa dibandingkan antar ruang? |
|---|---|---|
| `ari_boot` | ARI partisi subsample 80% terhadap partisi acuan, rata-rata 20 bootstrap | Ya |
| `rapuh` | jumlah cluster acuan dengan Jaccard bootstrap < 0,6 | Ya |
| `knn_top1` | porsi 10 tetangga terdekat yang berbagi kelas top-1 ImageNet (referensi independen) | Ya |
| `ari_seed` | ARI antar 5 seed KMeans | Ya |
| `silhouette` | silhouette cosine | Tidak, bergantung geometri ruang |

Aturan beku, dievaluasi pada K = 48: 7B dipakai bila `ari_boot` 7B minimal 0,03 di atas ViT-L, `knn_top1` tidak lebih rendah, dan `rapuh` tidak lebih banyak. Bila salah satu gagal, ViT-L dipakai karena lebih murah. Baseline lokal ViT-L pada K = 48 (resep CLS satu pandangan, `temp/eda_208`): `ari_boot` 0,638, `rapuh` 18 dari 48.

In [ ]:
def norml2(a):
    return a / np.linalg.norm(a, axis=1, keepdims=True)


def resep(e):
    return norml2(np.hstack([norml2(e['asli_cls'] + e['flip_cls']), norml2(e['asli_patch'] + e['flip_patch'])]))


def siapkan(x):
    return norml2(PCA(CFG.PCA_DIM, random_state=CFG.SEED).fit_transform(x))


def km(z, k, seed, n_init=4):
    return KMeans(k, n_init=n_init, random_state=seed).fit(z)


def jaccard_cluster(acuan, lab, idx):
    out = []
    for c in np.unique(acuan):
        a = set(idx[acuan[idx] == c])
        out.append(max(len(a & set(idx[lab[idx] == d])) / len(a | set(idx[lab[idx] == d])) for d in np.unique(lab[idx])))
    return np.array(out)


def metrik_ruang(z, k):
    acuan = km(z, k, CFG.SEED).labels_
    rng = np.random.default_rng(CFG.SEED)
    ari_seed = np.mean([adjusted_rand_score(acuan, km(z, k, CFG.SEED + s).labels_) for s in range(1, 5)])
    boot, jac = [], []
    for b in range(CFG.N_BOOT):
        idx = np.sort(rng.choice(len(z), int(.8 * len(z)), replace=False))
        lab = km(z[idx], k, CFG.SEED + 100 + b).predict(z)
        boot.append(adjusted_rand_score(acuan, lab))
        jac.append(jaccard_cluster(acuan, lab, idx))
    jac = np.mean(jac, 0)
    S = z @ z.T
    np.fill_diagonal(S, -np.inf)
    nn10 = np.argsort(-S, 1)[:, :10]
    uk = np.bincount(acuan)
    return dict(k=k, silhouette=silhouette_score(z, acuan, metric='cosine'), ari_seed=ari_seed, ari_boot=np.mean(boot),
                ari_boot_std=np.std(boot), jaccard_med=np.median(jac), rapuh=int((jac < .6).sum()),
                knn_top1=np.mean(TOP1[nn10] == TOP1[:, None]), uk_min=int(uk.min()), uk_max=int(uk.max()))


Z = {'vitl': siapkan(resep(E_VITL)), '7b': siapkan(resep(E_7B))}
banding = pd.DataFrame([dict(ruang=r, **metrik_ruang(z, CFG.K_BANDING)) for r, z in Z.items()]).set_index('ruang')
display(banding.round(3))
a, b = banding.loc['7b'], banding.loc['vitl']
syarat = {'ari_boot naik >= 0,03': a.ari_boot - b.ari_boot >= CFG.SELISIH_ARI_7B,
          'knn_top1 tidak turun': a.knn_top1 >= b.knn_top1, 'rapuh tidak bertambah': a.rapuh <= b.rapuh}
PAKAI = '7b' if all(syarat.values()) else 'vitl'
print(syarat)
print('ARI partisi K=48 antara 7B dan ViT-L:',
      round(adjusted_rand_score(km(Z['7b'], CFG.K_BANDING, CFG.SEED).labels_, km(Z['vitl'], CFG.K_BANDING, CFG.SEED).labels_), 3))
lama = cari_cache('label_lama.csv')
if lama:
    lab_lama = pd.read_csv(lama).sort_values('image_id').label.values
    for r, z in Z.items():
        print(f'ARI {r} K=48 vs label lama (melingkar untuk ViT-L):',
              round(adjusted_rand_score(lab_lama, km(z, CFG.K_BANDING, CFG.SEED).labels_), 3))
print('RUANG DIPAKAI:', PAKAI)

#### Insights

> Isi setelah run: tulis angka `ari_boot`, `rapuh`, dan `knn_top1` kedua ruang beserta ruang yang terpilih. Bila 7B tidak lolos, ekstraksi 7B tetap tercatat di notebook sebagai bukti bahwa keputusan diambil dari metrik.

---

# Clustering Train <a name="6"></a>

---

## Pemilihan K dan Partisi Acuan

Train sengaja di-overcluster (K lebih besar dari 8) supaya tiap cluster cukup murni untuk diberi satu nama. K dipilih dari {32, 48, 64} dengan `ari_boot` tertinggi, dengan syarat cluster terkecil minimal 10 citra. Partisi acuan memakai KMeans `n_init=20`, lalu nomor cluster diurutkan menurut ID citra terkecil di dalamnya, sehingga nomornya tidak bergantung pada urutan label internal KMeans. Partisi disimpan ke `cluster_train.csv`. Bila cache ada, partisi dimuat dari cache supaya nama cluster di bagian 7 tetap menunjuk citra yang sama.

In [ ]:
def cluster_acuan(z, k):
    lab = km(z, k, CFG.SEED, n_init=20).labels_
    urut = np.argsort([np.flatnonzero(lab == c).min() for c in range(k)])
    peta = np.empty(k, int)
    peta[urut] = np.arange(k)
    return peta[lab]


def peta_bootstrap(z, acuan, k):
    rng = np.random.default_rng(CFG.SEED + 7)
    out = []
    for b in range(CFG.N_BOOT):
        idx = np.sort(rng.choice(len(z), int(.8 * len(z)), replace=False))
        lab = km(z[idx], k, CFG.SEED + 200 + b).predict(z)
        out.append(pd.crosstab(lab, acuan).idxmax(axis=1).loc[lab].values)
    return np.array(out)


ZP = Z[PAKAI]
cache_cl = cari_cache('cluster_train.csv')
if CFG.PAKAI_CACHE and cache_cl:
    CL = pd.read_csv(cache_cl)
    assert CL.ruang.iloc[0] == PAKAI, f'cache cluster dari ruang {CL.ruang.iloc[0]}, run ini memilih {PAKAI}'
    K = int(CL.cluster.max() + 1)
    BOOT = np.load(cari_cache('boot_train.npy'))
    print('partisi dimuat dari cache; ARI dengan hitung ulang:', round(adjusted_rand_score(CL.cluster, cluster_acuan(ZP, K)), 4))
else:
    kandidat = pd.DataFrame([metrik_ruang(ZP, k) for k in CFG.K_KANDIDAT]).set_index('k')
    display(kandidat.round(3))
    lolos = kandidat[kandidat.uk_min >= CFG.UK_MIN_CLUSTER]
    K = int(lolos.ari_boot.idxmax()) if len(lolos) else CFG.K_BANDING
    acuan = cluster_acuan(ZP, K)
    BOOT = peta_bootstrap(ZP, acuan, K)
    CL = pd.DataFrame({'image_id': id_train, 'cluster': acuan, 'stab': (BOOT == acuan).mean(0), 'ruang': PAKAI})
    CL.to_csv(CFG.CACHE_DIR / 'cluster_train.csv', index=False)
    np.save(CFG.CACHE_DIR / 'boot_train.npy', BOOT)
print(f'K = {K} | citra stabil di level cluster (stab >= {CFG.STAB_MIN}): {(CL.stab >= CFG.STAB_MIN).mean():.3f}')
fig, ax = plt.subplots(1, 2, figsize=(13, 3.5))
CL.cluster.value_counts().sort_index().plot.bar(ax=ax[0], fontsize=6)
ax[0].set(title='ukuran cluster', xlabel='cluster')
ax[1].hist(CL.stab, 20)
ax[1].axvline(CFG.STAB_MIN, c='k', ls='--')
ax[1].set(title='stabilitas per citra (bootstrap)')
plt.tight_layout()
plt.show()

#### Insights

> Pada ViT-L lokal, sekitar separuh cluster rapuh untuk K 32 sampai 64 (Jaccard bootstrap < 0,6), artinya batas cluster bergeser antar run. Matriks pemetaan 20 bootstrap disimpan (`boot_train.npy`). Setelah penamaan, label hanya diberikan pada citra yang jatuh ke cluster bernama sama pada minimal 70% bootstrap (`stab_nama`). Citra yang hanya berpindah di antara dua cluster bernama sama tetap berlabel. Ukuran cluster ViT-L K = 48 berkisar 15 sampai 83 citra (rasio 5,5), jadi imbalance antar cluster jauh lebih besar daripada imbalance antar kelas pada label lama (rasio 1,49).

## Grup Near-duplicate

Grup untuk validasi dibentuk dari train saja dengan union-find: pasangan pHash dihedral Hamming 4 atau kurang, atau pasangan dengan cosine embedding tertinggi. Ambang cosine diambil dari kuantil supaya kepadatan pasangannya sama dengan cos 0,92 pada ViT-L di EDA lokal (0,0721% dari semua pasangan). Alasannya, nilai cosine mentah tidak setara antar ruang embedding.

In [ ]:
EG = resep(E_7B if PAKAI == '7b' else E_VITL)
C = EG @ EG.T
ambang_cos = np.sort(C[iu])[::-1][int(CFG.PORSI_PASANGAN * len(iu[0]))]
_, GRUP = connected_components(csr_matrix((HAM <= CFG.H_MAX) | (C >= ambang_cos)), directed=False)
uk_grup = np.bincount(GRUP)
CL['grup'] = GRUP
print(f'ambang cos {ambang_cos:.4f} | {len(uk_grup)} grup | terbesar {uk_grup.max()} | citra dalam grup > 1: {int((uk_grup[GRUP] > 1).sum())}')
del C

#### Insights

> Di EDA lokal, gabungan Hamming 4 dan cos 0,92 menghasilkan 1.471 grup dengan grup terbesar 52 citra (2,7% data), dan 628 citra berada dalam grup berisi lebih dari satu. Grup sebesar ini masih bisa dibagi rata oleh StratifiedGroupKFold 5 fold tanpa merusak stratifikasi.

---

# Penamaan Cluster <a name="7"></a>

---

Penamaan dilakukan manual oleh tim, sesuai jawaban resmi panitia. Untuk tiap cluster, notebook menampilkan montage berisi 20 citra inti (stabilitas tertinggi, diambil acak) dan 6 citra paling pinggir (bingkai merah), serta tabel ukuran, stabilitas rata-rata, dan top-5 kelas ImageNet. Skor head hanya bukti pendukung. Keputusan nama ada pada tim, dan cluster campuran boleh dipecah atau ditandai `campur`. Asisten AI yang membantu menulis kode hanya melihat grid preprocessing untuk QA, dan tidak memberi label maupun nama cluster.

## Bukti Visual per Cluster

Montage juga disimpan sebagai `gambar/cluster_<kode>.png` supaya bisa diperiksa di luar notebook.

In [ ]:
def urut_kode(kode):
    return sorted(set(kode), key=lambda s: (int(''.join(c for c in s if c.isdigit())), s))


def top_in(idx, n=5):
    p = P_HEAD[idx].mean(0)
    return ', '.join(f"{NAMA_IN[i].split(',')[0]} {p[i]:.2f}" for i in np.argsort(-p)[:n])


def montage(kd, inti, pinggir):
    idx = list(inti) + list(pinggir)
    fig, ax = plt.subplots(2, 13, figsize=(16, 2.9))
    for a in ax.ravel():
        a.axis('off')
    for j, (a, i) in enumerate(zip(ax.ravel(), idx)):
        a.imshow(cv2.resize(X_train[i], (112, 112), interpolation=cv2.INTER_AREA))
        a.set_title(f'{id_train[i]} {CL.stab[i]:.2f}', fontsize=6, color='r' if j >= len(inti) else 'k')
    fig.suptitle(f'cluster {kd} (n={int((CL.kode == kd).sum())}) | merah = paling pinggir', fontsize=9)
    fig.tight_layout()
    fig.savefig(CFG.FIG_DIR / f'cluster_{kd}.png', dpi=80)
    plt.show()


def bukti(daftar_kode):
    rows = []
    rng = np.random.default_rng(CFG.SEED)
    for kd in daftar_kode:
        d = CL[CL.kode == kd].sort_values('stab', ascending=False)
        inti = rng.permutation(d.index.values[:max(20, len(d) // 2)])[:20]
        pinggir = d.index.values[::-1][:min(6, max(0, len(d) - len(inti)))]
        montage(kd, inti, pinggir)
        rows.append(dict(kode=kd, n=len(d), stab_mean=round(d.stab.mean(), 3), n_pinggir=int((d.stab < CFG.STAB_MIN).sum()),
                         top5_imagenet=top_in(d.index.values)))
    return pd.DataFrame(rows)


CL['kode'] = CL.cluster.astype(str)
BUKTI = bukti(urut_kode(CL.kode))
BUKTI.to_csv(CFG.OUTPUT_DIR / 'bukti_cluster.csv', index=False)
with pd.option_context('display.max_colwidth', 200, 'display.max_rows', 200):
    display(BUKTI)

## Pecah Cluster Campuran

Bila montage sebuah cluster memuat dua material atau lebih, cluster itu bisa dipecah dengan KMeans lokal. Contoh: `PECAH = {12: 2}` memecah cluster 12 menjadi `12a` dan `12b`, lalu montage keduanya ditampilkan. Biarkan kosong bila tidak ada cluster yang perlu dipecah.

In [ ]:
PECAH = {}

CL['kode'] = CL.cluster.astype(str)
for c, n in PECAH.items():
    idx = CL.index[CL.cluster == c]
    sub = KMeans(n, n_init=10, random_state=CFG.SEED).fit_predict(ZP[idx])
    CL.loc[idx, 'kode'] = [f'{c}{chr(97 + s)}' for s in sub]
if PECAH:
    display(bukti(urut_kode(CL.kode[CL.cluster.isin(list(PECAH))])))
print('templat NAMA_CLUSTER:')
print('{' + ', '.join(f"'{k}': ''" for k in urut_kode(CL.kode)) + '}')

## Nama Cluster dari Tim

Isi `NAMA_CLUSTER` dengan salah satu nama kelas (`glass`, `paper`, `electronic`, `metal`, `plastic`, `cardboard`, `textile`, `foodwaste`) atau `campur` untuk setiap kode, berdasarkan montage di atas. Salin templat dari sel sebelumnya. Citra pada cluster `campur` tidak diberi label. Selama kamus ini belum lengkap, run tetap selesai normal: sel berikutnya mencetak pesan tahap 1, dan bagian 8 sampai 11 dilewati supaya cache serta montage tetap tersimpan sebagai output.

In [ ]:
NAMA_CLUSTER = {
}

In [ ]:
belum = [k for k in urut_kode(CL.kode) if k not in NAMA_CLUSTER]
LABEL_SIAP = not belum
if not LABEL_SIAP:
    print('TAHAP 1 SELESAI: NAMA_CLUSTER belum diisi. Lihat gambar/cluster_*.png dan bukti_cluster.csv, isi nama, lalu jalankan ulang dengan cache.')
else:
    salah = {k: v for k, v in NAMA_CLUSTER.items() if v not in CFG.KELAS + ['campur']}
    assert not salah, f'nama tidak valid: {salah}'
    CL['nama'] = CL.kode.map(NAMA_CLUSTER)
    nama_cluster = CL.groupby('cluster').nama.agg(lambda s: s.mode()[0]).reindex(range(K)).to_numpy(dtype=object)
    # cluster yang dipecah (PECAH) bisa berisi beberapa nama; jatuh ke cluster sendiri tetap dihitung konsisten
    CL['stab_nama'] = ((nama_cluster[BOOT] == CL.nama.to_numpy(dtype=object)) | (BOOT == CL.cluster.values)).mean(0)
    CL['berlabel'] = CL.nama.isin(CFG.KELAS) & (CL.stab_nama >= CFG.STAB_MIN)
    CL.to_csv(CFG.OUTPUT_DIR / 'label_train.csv', index=False)
    L = np.flatnonzero(CL.berlabel.values)
    Y = np.array([CFG.KELAS.index(n) for n in CL.nama.values[L]])
    print(f'berlabel {len(L)} dari 1.920 | campur {int((CL.nama == "campur").sum())} | ambigu antar nama {int((CL.stab_nama < CFG.STAB_MIN).sum())}')
    display(pd.Series(Y).map(dict(enumerate(CFG.KELAS))).value_counts().rename('n').to_frame().T)

    rng = np.random.default_rng(CFG.SEED)
    fig, ax = plt.subplots(8, 10, figsize=(13, 11))
    for r, kls in enumerate(CFG.KELAS):
        pilih = rng.permutation(L[Y == r])[:10]
        for c in range(10):
            ax[r, c].axis('off')
            if c < len(pilih):
                ax[r, c].imshow(cv2.resize(X_train[pilih[c]], (96, 96), interpolation=cv2.INTER_AREA))
        ax[r, 0].set_title(kls, fontsize=8, loc='left')
    plt.tight_layout()
    plt.show()

#### Insights

> Isi setelah penamaan: jumlah citra berlabel per kelas, cluster yang ditandai `campur` beserta alasannya, dan cluster yang nama dari montage berbeda dari top-5 ImageNet.

---

# Training Classifier <a name="8"></a>

---

Classifier final adalah ConvNeXt-L DINOv3 yang di-fine-tune pada piksel, bukan head di atas embedding 7B. Ada tiga alasan. Pertama, head linear pada ruang yang sama dengan ruang clustering hanya mereproduksi batas cluster, termasuk kesalahannya, dan skor validasinya melingkar. Kedua, uji lama menunjukkan fine-tune menambah informasi di atas head linear. Ketiga, inference test tidak membutuhkan 7B sehingga notebook final lebih ringan.

Resep mengikuti EXP-010 yang sudah teruji di data ini: AdamW, LR 1e-4 dengan pengali head 10, layer decay 0,75, weight decay 0,05, warmup 1 epoch lalu cosine, label smoothing 0,1, drop path 0,1, bobot kelas akar-invers frekuensi, dan 8 epoch. Karena T4 tidak punya bf16, training memakai AMP fp16 dengan GradScaler. Batch 16 dibagi ke dua GPU lewat DataParallel.

In [ ]:
def buat_cnn():
    return timm.create_model(CFG.MCNN[2], pretrained=True, num_classes=len(CFG.KELAS), drop_path_rate=CFG.DROP_PATH,
                             pretrained_cfg_overlay={'file': str(unduh(*CFG.MCNN[:2]))})


def bungkus(model):
    return nn.DataParallel(model) if torch.cuda.device_count() > 1 else model


def bobot_kelas(y):
    w = 1 / np.sqrt(np.bincount(y, minlength=len(CFG.KELAS)).clip(1))
    return torch.tensor(w / w.mean(), dtype=torch.float32)


@torch.no_grad()
def prediksi(net, X, idx=None):
    net.eval()
    dl = DataLoader(Citra(X, tf_eval, idx), batch_size=32, num_workers=CFG.NUM_WORKERS)
    out = []
    with torch.autocast('cuda', dtype=torch.float16):
        for x in dl:
            out.append(torch.softmax(net(x.cuda().to(memory_format=torch.channels_last)).float(), 1).cpu())
    return torch.cat(out).numpy()


def latih(idx_tr, y_tr, idx_va=None, y_va=None):
    seed_everything(CFG.SEED)
    model = buat_cnn().cuda().to(memory_format=torch.channels_last)
    opt = create_optimizer_v2(model, opt='adamw', lr=CFG.LR, weight_decay=CFG.WD, layer_decay=CFG.LAYER_DECAY)
    puncak = max(g.get('lr_scale', 1.0) for g in opt.param_groups)
    for g in opt.param_groups:
        g['lr_scale'] = g.get('lr_scale', 1.0) * (CFG.HEAD_MULT if g.get('lr_scale', 1.0) >= puncak else 1.0)
    net = bungkus(model)
    dl = DataLoader(Citra(X_train, tf_train, idx_tr, y_tr), batch_size=CFG.BS, shuffle=True, drop_last=True,
                    num_workers=CFG.NUM_WORKERS, worker_init_fn=seed_worker,
                    generator=torch.Generator().manual_seed(CFG.SEED), persistent_workers=True)
    total, warm = len(dl) * CFG.EPOCH, len(dl) * CFG.WARMUP
    kriteria = nn.CrossEntropyLoss(weight=bobot_kelas(y_tr).cuda(), label_smoothing=CFG.SMOOTH)
    scaler = torch.amp.GradScaler('cuda')
    riwayat, langkah, pv = [], 0, None
    for ep in range(CFG.EPOCH):
        net.train()
        rugi = []
        t0 = time.time()
        for x, y in dl:
            if langkah < warm:
                f = langkah / warm
            else:
                f = 0.5 * (1 + math.cos(math.pi * (langkah - warm) / max(1, total - warm)))
            for g in opt.param_groups:
                g['lr'] = CFG.LR * g['lr_scale'] * f
            with torch.autocast('cuda', dtype=torch.float16):
                loss = kriteria(net(x.cuda(non_blocking=True).to(memory_format=torch.channels_last)), y.cuda())
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()
            rugi.append(loss.item())
            langkah += 1
        r = dict(epoch=ep + 1, loss=round(float(np.mean(rugi)), 4), detik=round(time.time() - t0))
        if idx_va is not None:
            pv = prediksi(net, X_train, idx_va)
            r['val_acc'] = round(float((pv.argmax(1) == y_va).mean()), 4)
        riwayat.append(r)
        print(r, flush=True)
    return model, riwayat, pv

## Cross-validation

StratifiedGroupKFold 5 fold memakai grup near-duplicate, sehingga salinan sebuah citra tidak muncul di train dan validasi sekaligus. Hasil OOF disimpan ke `cache/cv_oof.npz`. Cache dipakai ulang hanya bila himpunan label sama persis dan `ULANG_CV = False`.

In [ ]:
if not LABEL_SIAP:
    print('Dilewati: NAMA_CLUSTER belum diisi (tahap 1).')
else:
    sgkf = StratifiedGroupKFold(n_splits=CFG.N_FOLD, shuffle=True, random_state=CFG.SEED)
    FOLD = np.full(len(L), -1)
    for f, (_, b) in enumerate(sgkf.split(L, Y, GRUP[L])):
        FOLD[b] = f
    cache_cv = cari_cache('cv_oof.npz')
    cv = np.load(cache_cv) if cache_cv and not CFG.ULANG_CV else None
    if cv is not None and np.array_equal(cv['L'], L) and np.array_equal(cv['Y'], Y) and np.array_equal(cv['fold'], FOLD):
        OOF, RIWAYAT = cv['oof'], json.loads(str(cv['riwayat']))
        print('OOF dimuat dari cache')
    else:
        OOF, RIWAYAT = np.zeros((len(L), len(CFG.KELAS))), []
        for f in range(CFG.N_FOLD):
            a, b = np.flatnonzero(FOLD != f), np.flatnonzero(FOLD == f)
            print(f'fold {f}: latih {len(a)}, validasi {len(b)}')
            model, riw, pv = latih(L[a], Y[a], L[b], Y[b])
            OOF[b] = pv
            RIWAYAT += [dict(fold=f, **r) for r in riw]
            del model
            gc.collect()
            torch.cuda.empty_cache()
        np.savez(CFG.CACHE_DIR / 'cv_oof.npz', oof=OOF, L=L, Y=Y, fold=FOLD, riwayat=json.dumps(RIWAYAT))

---

# Validasi Lokal <a name="9"></a>

---

Accuracy di sini adalah kesepakatan model dengan label cluster yang dinamai tim, bukan accuracy terhadap ground truth. Simpangan baku antar fold menunjukkan seberapa besar skor bisa bergeser hanya karena sampel yang berbeda. Sebagai pembanding, galat baku binomial pada accuracy 0,95 dengan 145 citra (public) sekitar 1,8 poin persen, dan dengan 335 citra (private) sekitar 1,2 poin persen.

In [ ]:
if not LABEL_SIAP:
    print('Dilewati: NAMA_CLUSTER belum diisi (tahap 1).')
else:
    acc = np.array([(OOF[FOLD == f].argmax(1) == Y[FOLD == f]).mean() for f in range(CFG.N_FOLD)])
    print('accuracy per fold:', acc.round(4))
    print(f'accuracy CV: {acc.mean():.4f} +- {acc.std(ddof=1):.4f} (mean +- std antar fold)')
    for n in (145, 335):
        print(f'galat baku binomial pada n={n}: {math.sqrt(acc.mean() * (1 - acc.mean()) / n):.4f}')
    pred = OOF.argmax(1)
    cm = confusion_matrix(Y, pred, labels=range(len(CFG.KELAS)))
    recall = pd.Series(cm.diagonal() / cm.sum(1), CFG.KELAS).round(3)
    display(recall.to_frame('recall OOF').T)
    riw = pd.DataFrame(RIWAYAT)
    fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
    im = ax[0].imshow(cm, cmap='Blues')
    ax[0].set_xticks(range(8), CFG.KELAS, rotation=45, fontsize=7)
    ax[0].set_yticks(range(8), CFG.KELAS, fontsize=7)
    for i in range(8):
        for j in range(8):
            ax[0].text(j, i, cm[i, j], ha='center', va='center', fontsize=7)
    ax[0].set(xlabel='prediksi OOF', ylabel='label cluster', title='confusion matrix OOF')
    for f, d in riw.groupby('fold'):
        ax[1].plot(d.epoch, d.val_acc, marker='o', label=f'fold {f}')
    ax[1].set(xlabel='epoch', ylabel='accuracy validasi', title='kurva validasi per fold')
    ax[1].legend()
    plt.tight_layout()
    plt.show()

    beda = CL.iloc[L].assign(pred=np.array(CFG.KELAS)[pred], benar=pred == Y)
    tinjau = beda.groupby('kode').agg(nama=('nama', 'first'), n=('benar', 'size'), setuju=('benar', 'mean'),
                                      pred_lain=('pred', lambda s: s.value_counts().index[0]))
    display(tinjau[tinjau.setuju < .9].sort_values('setuju').round(3))

#### Insights

> Isi setelah run: mean +- std accuracy per fold, kelas dengan recall terendah, dan cluster di tabel terakhir. Cluster dengan kesepakatan di bawah 0,9 adalah kandidat nama yang perlu ditinjau ulang lewat montage-nya. Penggantian nama tetap keputusan tim berdasarkan citra train. Bila std antar fold lebih besar dari sekitar 1,5 poin persen, selisih skor public dan private yang besar wajar terjadi.

---

# Inference Test <a name="10"></a>

---

Model final dilatih ulang pada seluruh citra berlabel dengan resep dan jumlah epoch yang sama dengan cross-validation. Model fold tidak dirata-rata karena itu termasuk ensemble. Test baru dimuat di sini, dengan preprocessing yang identik dengan validasi (stretch ke 384, tanpa TTA), lalu diprediksi satu kali.

In [ ]:
if not LABEL_SIAP:
    print('Dilewati: NAMA_CLUSTER belum diisi (tahap 1).')
else:
    MODEL, RIW_FINAL, _ = latih(L, Y)
    with ThreadPoolExecutor(8) as ex:
        X_test = np.stack(list(ex.map(muat_citra, f_test)))
    P_TEST = prediksi(bungkus(MODEL), X_test)
    np.save(CFG.CACHE_DIR / 'prob_test.npy', P_TEST.astype(np.float16))
    print('distribusi prediksi test:', pd.Series(np.array(CFG.KELAS)[P_TEST.argmax(1)]).value_counts().to_dict())

---

# Submission <a name="11"></a>

---

Berkas submission mengikuti format `image_id,waste_category` dengan ID tiga digit, sesuai sample submission. Urutan baris disamakan dengan sample submission.

In [ ]:
if not LABEL_SIAP:
    print('Dilewati: NAMA_CLUSTER belum diisi (tahap 1).')
else:
    contoh = pd.read_csv(SUB_PATH, dtype=str)
    pred = pd.DataFrame({'image_id': [f'{i:03d}' for i in id_test], 'waste_category': np.array(CFG.KELAS)[P_TEST.argmax(1)]})
    assert set(pred.image_id) == set(contoh.image_id)
    submission = contoh[['image_id']].merge(pred, on='image_id', how='left')
    assert submission.waste_category.isin(CFG.KELAS).all() and len(submission) == 480
    submission.to_csv(CFG.OUTPUT_DIR / 'submission.csv', index=False)
    print(submission.waste_category.value_counts().to_dict())
    display(submission.head())